# 02 - Exploratory Data Analysis & Business Intelligence
### Project: E-Commerce Customer, Sales & Business Intelligence Analytics
**Author:** Senior Data Analyst & BI Developer  
**Dataset:** Cleaned Brazilian Olist E-Commerce Dataset  

---

## 1. Objectives
This notebook performs statistical analysis and business intelligence investigation:
1. **Executive KPIs**: Gross Revenue (Product + Freight), Total Orders, AOV, Delivery Rate, Customer Satisfaction.
2. **Sales & Category Dynamics**: Revenue and item volume across product categories and price points.
3. **Customer Intelligence**: Retention metrics (New vs Repeat Customers) and Customer Lifetime Value (CLV) Segmentation.
4. **Delivery Performance & Operational Efficiency**: Lead-time distributions and fulfillment analysis.
5. **Correlation Investigation**: Empirical analysis of delivery lead time vs customer review scores.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', None)

PROCESSED_DATA_DIR = '../data/processed'

df_master = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'analytics_master_orders.csv'))
df_items = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'fact_order_items_clean.csv'))
df_products = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'dim_products_clean.csv'))
df_customers = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'dim_customers_clean.csv'))
df_reviews = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'fact_order_reviews_clean.csv'))

df_items_prod = df_items.merge(df_products[['product_id', 'product_category_name_english']], on='product_id', how='left')
print("Datasets loaded successfully.")


## 2. Core Executive KPIs

In [ ]:
tot_revenue = df_items['revenue'].sum()
tot_orders = df_master['order_id'].nunique()
tot_customers = df_customers['customer_unique_id'].nunique()
aov = tot_revenue / tot_orders
delivered_rate = (df_master['order_status'] == 'delivered').mean() * 100
avg_review = df_reviews['review_score'].mean()
avg_delivery = df_master[df_master['order_status'] == 'delivered']['delivery_days'].mean()

kpi_table = pd.DataFrame({
    'Metric': ['Gross Revenue', 'Total Orders', 'Unique Customers', 'Average Order Value (AOV)', 'Delivery Rate', 'Average Review Score', 'Average Delivery Time'],
    'Value': [f"R$ {tot_revenue:,.2f}", f"{tot_orders:,}", f"{tot_customers:,}", f"R$ {aov:.2f}", f"{delivered_rate:.2f}%", f"{avg_review:.2f} / 5.0", f"{avg_delivery:.1f} days"]
})
kpi_table


## 3. Monthly Revenue & Order Growth Trend

In [ ]:
monthly_trend = df_master[df_master['order_year_month'] >= '2017-01'].groupby('order_year_month').agg(
    Revenue=('total_order_revenue', 'sum'),
    Orders=('order_id', 'nunique')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(monthly_trend['order_year_month'], monthly_trend['Revenue'] / 1e3, color='#2B6CB0', marker='o', linewidth=2.5, label='Revenue (k BRL)')
ax2.bar(monthly_trend['order_year_month'], monthly_trend['Orders'], alpha=0.3, color='#4A5568', label='Orders')

ax1.set_title("Monthly Revenue & Order Volume Growth (2017 - 2018)", fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel("Year-Month")
ax1.set_ylabel("Revenue (Thousands BRL)", color='#2B6CB0')
ax2.set_ylabel("Order Count", color='#4A5568')
ax1.set_xticklabels(monthly_trend['order_year_month'], rotation=45, ha='right')
plt.show()


## 4. Category Sales & Pricing Structure

In [ ]:
cat_summary = df_items_prod.groupby('product_category_name_english').agg(
    Revenue=('revenue', 'sum'),
    Items_Sold=('order_item_id', 'count'),
    Avg_Price=('price', 'mean')
).sort_values('Revenue', ascending=False)

top10 = cat_summary.head(10)
plt.figure(figsize=(10, 5))
sns.barplot(data=top10.reset_index(), y='product_category_name_english', x='Revenue', palette='Blues_r', hue='product_category_name_english', legend=False)
plt.title("Top 10 Product Categories by Gross Revenue (BRL)", fontsize=13, fontweight='bold')
plt.xlabel("Revenue (BRL)")
plt.ylabel("")
plt.show()


## 5. Customer Retention & Value Segmentation

In [ ]:
cust_retention = df_customers.groupby('customer_unique_id')['customer_order_count'].first()
new_c = (cust_retention == 1).sum()
rep_c = (cust_retention > 1).sum()

plt.figure(figsize=(6, 6))
plt.pie([new_c, rep_c], labels=[f'One-Time ({new_c:,})', f'Repeat ({rep_c:,})'], autopct='%1.1f%%',
        colors=['#4299E1', '#48BB78'], startangle=140, wedgeprops=dict(width=0.45, edgecolor='white'))
plt.title("Customer Retention Proportion (New vs Repeat)", fontsize=13, fontweight='bold')
plt.show()


## 6. Correlation Analysis: Delivery Lead Time vs Review Rating

In [ ]:
deliv_rev = df_master[df_master['order_status'] == 'delivered'].dropna(subset=['delivery_days', 'review_score'])
corr = deliv_rev['delivery_days'].corr(deliv_rev['review_score'])
print(f"Pearson Correlation between Delivery Days and Review Score: {corr:.4f}")

deliv_rev['delivery_tier'] = pd.cut(
    deliv_rev['delivery_days'],
    bins=[0, 5, 10, 15, 20, 30, 60, 200],
    labels=['0-5 Days', '6-10 Days', '11-15 Days', '16-20 Days', '21-30 Days', '31-60 Days', '60+ Days']
)
tier_summary = deliv_rev.groupby('delivery_tier', observed=False).agg(
    Orders=('order_id', 'count'),
    Avg_Rating=('review_score', 'mean'),
    Pct_5_Star=('review_score', lambda x: (x == 5).mean() * 100),
    Pct_1_Star=('review_score', lambda x: (x == 1).mean() * 100)
).reset_index()

print("\nDelivery Days Tier vs Customer Satisfaction:")
tier_summary
